# Publication snapshot v0.4.0
整理自既有研究；已移除outputs、憑證與私人路徑。見README與docs/limitations.md。
保留歷史實驗設計供檢視，不代表已修正資料切分或已在乾淨環境重跑。


# 實驗 1 — 在 FaithBench 上比較「事實一致性偵測器」（含你的方法）

對應路線圖的 1~4 步，做成一個**小規模、可重複、輸出 CSV** 的偵測比較實驗：

| 步驟 | 這個 notebook 做了什麼 |
|---|---|
| 1. 換真實資料 | 載入 **FaithBench**（真實 abstractive 摘要 + 人工標籤；二元：Unwanted=幻覺） |
| 2. LLM-as-judge | `ollama_judge`：用你本機 ollama 模型一句話判斷 |
| 3. NLI baseline | `summac_nli`（選用，預設關）＋兩個免模型的 naive baseline |
| 4. 你的方法 | `yours_gpt4o_detect`：**直接重用你 two_step_clean 的 Detect 步驟（GPT-4o + 5 類錯誤）**，把 `result == "Has Error"` 對應成「幻覺」 |

**任務定義**：每個偵測器對 `(source, summary)` 輸出 0=一致 / 1=幻覺；和 FaithBench 人工標籤比，算 accuracy / balanced accuracy，並單獨看「灰色地帶」表現。

**安全性**：結果**逐行寫進 CSV**（`exp1_results_long.csv`），就算 kernel 中途掛掉/ipynb 損毀，已跑的部分都在 CSV 裡。最後再輸出 `exp1_metrics_summary.csv`。

> 沒有 ollama、沒有 OpenAI key、沒網路也能跑：會自動只用兩個 naive baseline 在內建樣本上跑完整流程，先驗證管線。把對應開關打開就逐步接上真模型與真資料。


## 0. 設定（所有開關都在這格）

In [ ]:
import os, re, json, csv, urllib.request
from collections import Counter

# ---- 資料 ----
USE_FAITHBENCH   = True            # True：從 GitHub 下載 FaithBench（你的電腦有網路就會成功）
FAITHBENCH_BATCHES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16]           # 要用哪些 batch（1..16，無 13）。多放幾個就更大規模
MAX_SAMPLES      = 750              # 小規模先設小一點（控制 API 花費/時間），None=全部

# ---- 偵測器開關 ----
USE_OLLAMA       = True
OLLAMA_URL       = "http://localhost:11434"
OLLAMA_MODEL     = "qwen3:4b"      # ollama list 查你有的
USE_YOURS = False  # Explicit opt-in for paid teacher API calls
YOURS_DETECTOR_MODEL = "gpt-4o"
USE_NLI          = False           # SummaC NLI baseline（需 pip install summac，會下載模型）

# ---- 輸出 ----
RESULTS_CSV      = "exp1_results_long.csv"
METRICS_CSV      = "exp1_metrics_summary.csv"

import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
print("設定完成。")


## 1. 資料：FaithBench（真實）+ 內建備援

二元標籤沿用官方 `binarize.py` 精神：**有 `Unwanted` 標註 → 幻覺(1)；否則 → 一致(0)**。
`is_gray = True` 代表只被標 Benign/Questionable（灰色地帶，連 SOTA 偵測器都吵架的那種）。

In [ ]:
TOY_DATA = [
    {"id":"t1","gold":0,"is_gray":False,"note":"Consistent",
     "source":"The Eiffel Tower is 330 metres tall and is located in Paris, France.",
     "summary":"The Eiffel Tower, located in Paris, stands 330 metres tall."},
    {"id":"t2","gold":1,"is_gray":False,"note":"Wrong Number",
     "source":"The Eiffel Tower is 330 metres tall and is located in Paris, France.",
     "summary":"The Eiffel Tower is 500 metres tall and is located in Paris."},
    {"id":"t3","gold":0,"is_gray":False,"note":"Consistent",
     "source":"Apollo 11 landed on the Moon in 1969. Neil Armstrong was the first person to walk on the lunar surface.",
     "summary":"Neil Armstrong was the first person to walk on the Moon, during Apollo 11 in 1969."},
    {"id":"t4","gold":1,"is_gray":False,"note":"Wrong Entity",
     "source":"Apollo 11 landed on the Moon in 1969. Neil Armstrong was the first person to walk on the lunar surface.",
     "summary":"Apollo 11 landed on the Moon in 1969, and Buzz Aldrin was the first to walk on it."},
    {"id":"t5","gold":0,"is_gray":False,"note":"Consistent",
     "source":"The company reported revenue of $2 billion in 2023, up 10% from the previous year.",
     "summary":"In 2023 the company's revenue rose 10% to $2 billion."},
    {"id":"t6","gold":1,"is_gray":False,"note":"Negation/Direction Flip",
     "source":"The company reported revenue of $2 billion in 2023, up 10% from the previous year.",
     "summary":"In 2023 the company's revenue fell 10% to $2 billion."},
    {"id":"t7","gold":0,"is_gray":True,"note":"Benign",
     "source":"Mount Everest is the highest mountain above sea level.",
     "summary":"Mount Everest, the highest mountain on Earth, attracts many climbers each year."},
    {"id":"t8","gold":1,"is_gray":False,"note":"Wrong Entity/Fact",
     "source":"Marie Curie won the Nobel Prize in Physics in 1903 and in Chemistry in 1911.",
     "summary":"Marie Curie won the Nobel Prize in Physics in 1903 and in Literature in 1911."},
]

def load_faithbench(batches):
    rows = []
    for b in batches:
        url = ("https://raw.githubusercontent.com/vectara/FaithBench/main/"
               f"data_for_release/batch_{b}.json")
        with urllib.request.urlopen(url, timeout=30) as r:
            obj = json.load(r)
        for s in obj["samples"]:
            labels = [l for a in s.get("annotations", []) for l in (a.get("label") or [])]
            has_unwanted = any("Unwanted" in l for l in labels)
            has_annot    = len(s.get("annotations", [])) > 0
            rows.append({
                "id": f"b{b}_{s['sample_id']}",
                "source": s["source"], "summary": s["summary"],
                "gold": 1 if has_unwanted else 0,
                "is_gray": has_annot and not has_unwanted,
                "note": ",".join(sorted(set(labels))) or "Consistent",
                "summarizer": s.get("metadata", {}).get("summarizer", ""),
            })
    return rows

data = None
if USE_FAITHBENCH:
    try:
        data = load_faithbench(FAITHBENCH_BATCHES)
        print(f"已載入 FaithBench {FAITHBENCH_BATCHES}：{len(data)} 筆")
    except Exception as e:
        raise RuntimeError("FaithBench download failed; no toy fallback for benchmark runs") from e
if data is None:
    data = [dict(d, summarizer="(toy)") for d in TOY_DATA]

if MAX_SAMPLES:
    data = data[:MAX_SAMPLES]
print(f"本次使用 {len(data)} 筆；幻覺={sum(d['gold'] for d in data)}，灰色地帶={sum(d['is_gray'] for d in data)}")

## 2. 偵測器們

每個偵測器 = 一個函式 `fn(source, summary) -> (pred 0/1, reason 字串)`。

- `naive_overlap`：詞彙重疊（很弱，抓不到細微改寫）。
- `naive_numcheck`：summary 出現了 source 沒有的數字 → 判幻覺（能抓 Wrong Number，抓不到 entity/negation）。
- `ollama_judge`：你本機 ollama 一句話判斷。
- `yours_gpt4o_detect`：**你 two_step_clean 的 Detect 步驟原封不動**（GPT-4o + 5 類錯誤 typology）。
- `summac_nli`：NLI-based baseline（選用）。

In [ ]:
# ---- 共用：句子切分 ----
def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p for p in parts if p]

# ---- naive baselines（永遠可用）----
def _tok(s): return set(re.findall(r"[a-z0-9]+", str(s).lower()))
def judge_overlap(source, summary, threshold=0.6):
    su, st = _tok(summary), _tok(source)
    if not su: return 0, "empty"
    ov = len(su & st) / len(su)
    return (1 if ov < threshold else 0), f"overlap={ov:.2f}"

def _nums(s):
    out = set()
    for m in re.findall(r"\d[\d,\.]*", str(s)):
        t = m.strip(",.").replace(",", "")   # 正規化：去尾逗號/句點、去千分位
        if t: out.add(t)
    return out
def judge_numcheck(source, summary):
    extra = _nums(summary) - _nums(source)
    return (1 if extra else 0), (f"unseen numbers={sorted(extra)}" if extra else "numbers ok")

# ---- ollama LLM-as-judge ----
def ollama_available():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

JUDGE_PROMPT = """You are a strict factual-consistency checker for summarization.
Decide whether EVERY claim in the SUMMARY is directly supported by the SOURCE.
First line: exactly one word, CONSISTENT or HALLUCINATED. Second line: one-sentence reason.

SOURCE:
{source}

SUMMARY:
{summary}
"""
def _parse_ch(text):
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S | re.I).upper()
    ih, ic = t.find("HALLUCINAT"), t.find("CONSISTENT")
    if ih == -1 and ic == -1: return 0
    if ih == -1: return 0
    if ic == -1: return 1
    return 1 if ih < ic else 0
def judge_ollama(source, summary):
    import ollama
    resp = ollama.chat(model=OLLAMA_MODEL, stream=False,
                       messages=[{"role":"user","content":JUDGE_PROMPT.format(source=source, summary=summary)}])
    out = resp["message"]["content"]
    return _parse_ch(out), " ".join(out.split())[:160]

# ---- 你的方法：two_step_clean 的 Detect 步驟（GPT-4o）----
# ============================================================
# 貼成「新的一格」，放在「## 2 偵測器們」那格之後、「## 3 建立清單」之前，
# 然後從這格往下重新執行即可。它只覆寫 detect_errors_prompt（中性、Benign-aware 版），
# 不動你的 key、不動其他程式。yours_gpt4o_detect 會自動改用這個新 prompt。
# ============================================================

detect_errors_prompt = """
You will be given a news ARTICLE (the ONLY ground truth) and a SUMMARY of that article.

INPUT FORMAT:
- The SUMMARY is a JSON object with a key "sentences": a list of strings (one sentence each).
- The index of each sentence (0, 1, 2, ...) is its sentence_index.

IMPORTANT — DO NOT ASSUME THE SUMMARY IS FLAWED:
- The summary MAY be fully faithful, or it MAY contain one or more factual errors.
- Your job is to JUDGE faithfulness, not to find errors at all costs.

COUNT a sentence as a factual error ONLY IF it:
- directly CONTRADICTS the article, OR
- asserts a specific fact (named entity, number/date/amount, event, or who-did-what-to-whom)
  that the article does NOT support.

DO NOT flag a sentence when it is merely:
- a paraphrase, compression, or an inference clearly entailed by the article,
- correct general/world knowledge that does NOT contradict the article
  (even if the article itself does not mention it),
- a difference in wording or style that does not change the facts.
When you are unsure whether something is genuinely unsupported or just a harmless rephrasing,
PREFER NOT to flag it.

ERROR TYPES (choose the single most important one per flagged sentence):
Wrong Entity, Wrong Number, Fabricated Fact, Negation Flip, Role Inversion.

OUTPUT FORMAT (STRICT JSON, nothing else):
- "result": "Has Error" if at least one sentence is a genuine factual error by the rules above,
  otherwise "No Error".
- "errors": a list of objects, each with:
    - "sentence_index": integer,
    - "error_type": one of the five types,
    - "explanation": short reason, citing what the article says vs. the summary,
    - "proposed_fix_sentence": a corrected version of that sentence, consistent with the article.
  If there are no errors, "errors" MUST be an empty list.

Do NOT output anything outside the JSON object.

ARTICLE:
{article}

SUMMARY JSON:
{flawed_summary_json}

Now judge the summary and return the JSON object.
"""
print("已套用 detect_errors_prompt v2（中性 / Benign-aware）。重新執行第 3、4、5、6 格即可。")
_openai_client = None
def _get_openai():
    global _openai_client
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    return _openai_client

def detect_errors(article, flawed_sentences, model=None):
    client = _get_openai()
    prompt = detect_errors_prompt.format(
        article=article,
        flawed_summary_json=json.dumps({"sentences": flawed_sentences}, ensure_ascii=False))
    resp = client.chat.completions.create(
        model=model or YOURS_DETECTOR_MODEL, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role":"system","content":"You are a precise factual error detector."},
                  {"role":"user","content":prompt}])
    return json.loads(resp.choices[0].message.content)

def judge_yours(source, summary):
    sents = split_sentences(summary)
    rep = detect_errors(source, sents)
    pred = 1 if rep.get("result") == "Has Error" else 0
    types = [e.get("error_type") for e in rep.get("errors", []) if isinstance(e, dict)]
    return pred, f"{len(types)} err {types}"

# ---- SummaC NLI baseline（選用）----
_summac = None
def _get_summac():
    global _summac
    if _summac is None:
        from summac.model_summac import SummaCZS
        _summac = SummaCZS(granularity="sentence", model_name="vitc", device="cpu")
    return _summac
def judge_summac(source, summary, threshold=0.5):
    score = _get_summac().score([source], [summary])["scores"][0]
    return (1 if score < threshold else 0), f"summac={score:.3f}"

print("偵測器定義完成。")

## 3. 建立可用偵測器清單

In [ ]:
USE_STUDENT = False  # Enable after running 02_train_qlora.ipynb
if USE_STUDENT:
    import torch, time
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    
    STUDENT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    ADAPTER_DIR   = "student_halluc_detector"
    SYS_STUDENT = "You are a strict factual-consistency checker. Given a SOURCE article and a SUMMARY, answer with exactly one word: CONSISTENT or HALLUCINATED."
    
    student_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    if student_tok.pad_token is None:
        student_tok.pad_token = student_tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    student_base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, quantization_config=bnb,
                                                         device_map="auto", torch_dtype=torch.bfloat16)
    student_model = PeftModel.from_pretrained(student_base, ADAPTER_DIR).eval()
    
    student_latencies = []
    @torch.no_grad()
    def judge_student(source, summary):
        msgs = [{"role":"system","content":SYS_STUDENT},
                {"role":"user","content":f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}\n\nIs the summary factually consistent with the source? Answer CONSISTENT or HALLUCINATED."}]
        prompt = student_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = student_tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(student_model.device)
        t0 = time.time()
        out = student_model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=student_tok.pad_token_id)
        dt = time.time() - t0
        student_latencies.append(dt)
        gen = student_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
        return (1 if "HALLUCINAT" in gen else 0), gen.strip()


In [ ]:
detectors = []
if USE_STUDENT:
    detectors.append(("student_qwen3-4b-qlora", judge_student))
detectors.append(("naive_overlap", judge_overlap))
detectors.append(("naive_numcheck", judge_numcheck))
if USE_OLLAMA and ollama_available():
    detectors.append((f"ollama_judge[{OLLAMA_MODEL}]", judge_ollama))
else:
    print("· ollama 未啟用/未連上 → 跳過 ollama_judge")
if USE_YOURS and os.environ.get("OPENAI_API_KEY"):
    detectors.append(("yours_gpt4o_detect", judge_yours))
else:
    print("· 未設定 OPENAI_API_KEY → 跳過 yours_gpt4o_detect（你的方法）")
if USE_NLI:
    try:
        _get_summac(); detectors.append(("summac_nli", judge_summac))
    except Exception as e:
        print("· SummaC 載入失敗 → 跳過。", e)

print("本次會跑的偵測器：", [n for n, _ in detectors])

## 4. 執行 → 逐行寫入 CSV（防中途崩潰）

In [ ]:
RERUN_DETECTORS = None  # Fresh evaluation; no implicit cache reuse
# 想全部重跑：設成 None

fields = ["id","summarizer","detector","gold","pred","correct","is_gray","note","reason"]

old_df = None
if os.path.exists(RESULTS_CSV):
    try:
        old_df = pd.read_csv(RESULTS_CSV)
    except Exception:
        old_df = None

rows = []
for name, fn in detectors:
    need_rerun = (RERUN_DETECTORS is None) or (name in RERUN_DETECTORS)
    cached = None
    if not need_rerun and old_df is not None:
        cached = old_df[old_df["detector"] == name]
        if len(cached) != len(data):          # 舊結果不完整/樣本數對不上 → 還是要重跑
            cached = None
    if cached is not None:
        print(f"· {name}：沿用舊結果（{len(cached)} 筆），不重新呼叫 API/模型")
        rows.extend(cached.to_dict("records"))
        continue
    print(f"· {name}：重新執行 {len(data)} 筆")
    for j, ex in enumerate(data):
        try:
            pred, reason = fn(ex["source"], ex["summary"])
        except Exception as e:
            pred, reason = -1, f"ERROR: {e}"
        row = {"id":ex["id"], "summarizer":ex.get("summarizer",""), "detector":name,
               "gold":ex["gold"], "pred":pred,
               "correct":int(pred==ex["gold"]) if pred in (0,1) else "",
               "is_gray":ex["is_gray"], "note":ex.get("note",""), "reason":reason}
        rows.append(row)
        if (j+1) % 50 == 0:
            print(f"    ...{j+1}/{len(data)}")

df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
print(f"完成，逐行結果已存到 {RESULTS_CSV}（{len(rows)} 列，含沿用的舊結果）")
df.head(12)


## 5. 指標彙整 → 輸出 metrics CSV

In [ ]:
# 重新從硬碟讀回 long CSV，確保指標與存檔 100% 一致（防同步截斷）
df = pd.read_csv(RESULTS_CSV)
exp_rows = len(data) * len(detectors)
if len(df) != exp_rows:
    print(f"⚠️ 列數不符：CSV {len(df)} 列，預期 {exp_rows}（{len(data)}樣本×{len(detectors)}偵測器）。可能被截斷。")
else:
    print(f"✓ 完整：{len(df)} 列 = {len(data)} 樣本 × {len(detectors)} 偵測器")

def metrics_for(sub):
    valid = sub[sub["pred"].isin([0, 1])]
    if len(valid) == 0:
        return None
    yt, yp = valid["gold"].tolist(), valid["pred"].tolist()
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {
        "n": len(valid),
        "accuracy": round(accuracy_score(yt, yp), 3),
        "balanced_accuracy": round(balanced_accuracy_score(yt, yp), 3),
        "recall_halluc": round(tp / (tp + fn), 3) if tp + fn else None,
        "precision": round(tp / (tp + fp), 3) if tp + fp else None,
        "specificity": round(tn / (tn + fp), 3) if tn + fp else None,
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "gray_n": int(sub["is_gray"].sum()),
        "gray_flagged": int(valid[valid["is_gray"]]["pred"].sum()),
        "n_failed": int((sub["pred"] == -1).sum()),
    }

summary = [{"detector": n, **metrics_for(df[df["detector"] == n])}
           for n, _ in detectors if metrics_for(df[df["detector"] == n])]
metrics_df = pd.DataFrame(summary).sort_values("balanced_accuracy", ascending=False)
metrics_df.to_csv(METRICS_CSV, index=False, encoding="utf-8-sig")
with open("exp1_summary.json", "w", encoding="utf-8") as f:
    json.dump({"n_samples": len(data),
               "n_hallucinated": int(df.drop_duplicates("id")["gold"].sum()),
               "detectors": summary}, f, ensure_ascii=False, indent=2)
print("已輸出", METRICS_CSV, "與 exp1_summary.json\n")
print("===== 指標總表（可直接複製貼回對話）=====")
print(metrics_df.to_string(index=False))
metrics_df

## 6. 快速錯誤分析（哪種錯誤類型最容易漏）

把每個偵測器漏掉（gold=1, pred=0）的案例，依 FaithBench 的 `note`（錯誤類型）統計——這直接回答「我的方法對哪類錯誤最弱」，是論文與面試最有料的一段。

In [ ]:
# 依 FaithBench 錯誤類型，算每個偵測器在各型別上的 recall（哪類錯誤最容易漏）
def norm_types(note):
    out = []
    for t in str(note).split(","):
        t = t.strip()
        if t and t.lower() != "consistent":
            out.append(t.split(".")[0])   # Unwanted.Intrinsic -> Unwanted
    return set(out) or {"(untyped)"}

bytype = []
gold1 = df[df["gold"] == 1]
all_types = sorted({t for note in gold1["note"] for t in norm_types(note)})
for name, _ in detectors:
    sub = df[df["detector"] == name]
    for t in all_types:
        rel = sub[(sub["gold"] == 1) & sub["note"].apply(lambda n: t in norm_types(n))]
        if len(rel):
            caught = int((rel["pred"] == 1).sum())
            bytype.append({"detector": name, "error_type": t, "n": len(rel),
                           "caught": caught, "missed": len(rel) - caught,
                           "recall": round(caught / len(rel), 2)})
bytype_df = pd.DataFrame(bytype)
bytype_df.to_csv("exp1_by_errortype.csv", index=False, encoding="utf-8-sig")
print("已輸出 exp1_by_errortype.csv（各偵測器 × 錯誤類型的 recall）\n")
print(bytype_df.to_string(index=False) if len(bytype_df) else "（沒有 gold=1 樣本可分析）")

## 7. 接下來

- **擴大規模**：`FAITHBENCH_BATCHES=[1,2,3,...]`、調大 `MAX_SAMPLES`。
- **接你的完整 pipeline**：這裡只用了你的 **Detect**。若要評估 detect→fix→verify 的「修正」效果，需要有「正確參考摘要」的資料（FaithBench 是偵測標竿、沒有 corrected 參考），那會是另一個實驗，建議用你自己的 CNNDM 合成資料或找帶 reference 的資料集。
- **把 CSV 給我**：把 `exp1_results_long.csv` 和 `exp1_metrics_summary.csv` 丟回來，我幫你做更深入的比較分析、畫圖、寫成結果敘述。
- **量化對照**：把 `OLLAMA_MODEL` 換成同模型的 Q4_K_M 量化版再跑一次，比較偵測準度與速度差異。
